In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
i = 5
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    i -= 1
    if i == 0: break
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/LICENSE
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/README.md


## Training part

In [ ]:
!uv pip install --upgrade wandb

In [5]:
import os
import glob
import random
import warnings
import multiprocessing
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
from torchmetrics.classification import MulticlassF1Score
from pathlib import Path
from tqdm import tqdm
import concurrent.futures

In [6]:
# ==========================================
# 0. CONFIGURATION & SETUP
# ==========================================
warnings.filterwarnings('ignore')
pl.seed_everything(14, workers=True)

# Toggle pipeline stages here
RUN_GENERATION = True
RUN_TRAINING = False
RUN_INFERENCE = False

CONFIG = {
    "project_name": "23f2001635-t12026",
    "experiment_name": "scratch_cnn_v1",
    "data_root": "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup",
    "save_dir": "/kaggle/working/processed_data",
    "lr": 1e-4,
    "batch_size": 16,
    "accumulate_grad": 4,
    "num_epochs": 100,
    "patience": 5,
    "num_classes": 10,
    "sample_rate": 16000,
    "duration": 30,  # Strict 30-second crops
    "train_samples": 3400,
    "val_samples": 600,
    "num_workers": multiprocessing.cpu_count(),
}

GENRES = ["blues", "classical", "country", "disco", "hiphop", 
          "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

# Target length for 30s @ 16000Hz, hop_length 512 = ~938 time steps
TARGET_TIME_STEPS = (CONFIG["duration"] * CONFIG["sample_rate"]) // 512 + 1 

# WandB Login (Only if training)
if RUN_TRAINING:
    import wandb
    from kaggle_secrets import UserSecretsClient
    try:
        user_secrets = UserSecretsClient()
        wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))
    except Exception as e:
        print(f"WandB login skipped/failed: {e}")

Seed set to 14


In [7]:
# ==========================================
# 1. DATA GENERATION (CPU MULTIPROCESSING)
# ==========================================

def get_random_stem(data_root, genre, stem_name, sample_rate):
    genre_dir = Path(data_root) / "genres_stems" / genre
    song_folders = [f.path for f in os.scandir(genre_dir) if f.is_dir()]
    random_song = random.choice(song_folders)
    stem_path = Path(random_song) / f"{stem_name}.wav"
    audio, _ = librosa.load(stem_path, sr=sample_rate)
    return audio

def get_random_noise(data_root, sample_rate):
    noise_dir = Path(data_root) / "ESC-50-master" / "audio"
    noise_files = glob.glob(str(noise_dir / "*.wav"))
    noise_path = random.choice(noise_files)
    noise_audio, _ = librosa.load(noise_path, sr=sample_rate)
    return noise_audio

def process_single_mashup(sample_id, genre, config):
    data_root = config["data_root"]
    sr = config["sample_rate"]
    target_samples = config["duration"] * sr
    
    try:
        # Load stems (Note: 'other.wav' based on your folder structure)
        drums = get_random_stem(data_root, genre, "drums", sr)
        bass = get_random_stem(data_root, genre, "bass", sr)
        vocals = get_random_stem(data_root, genre, "vocals", sr)
        other = get_random_stem(data_root, genre, "other", sr)

        # Tempo Matching (Anchor = drums)
        anchor_tempo, _ = librosa.beat.beat_track(y=drums, sr=sr)
        anchor_tempo = anchor_tempo[0] if isinstance(anchor_tempo, np.ndarray) else anchor_tempo
        
        stems = [drums]
        for track in [bass, vocals, other]:
            track_tempo, _ = librosa.beat.beat_track(y=track, sr=sr)
            track_tempo = track_tempo[0] if isinstance(track_tempo, np.ndarray) else track_tempo
            
            if track_tempo > 0 and 0.5 < (anchor_tempo / track_tempo) < 2.0:
                rate = anchor_tempo / track_tempo
                stems.append(librosa.effects.time_stretch(y=track, rate=rate))
            else:
                stems.append(track)

        # Pad/Crop stems & Mix
        max_len = min(max(len(s) for s in stems), target_samples)
        mixed_audio = np.zeros(max_len)
        for s in stems:
            s_adjusted = np.pad(s, (0, max(0, max_len - len(s))))[:max_len]
            mixed_audio += s_adjusted * random.uniform(0.5, 1.5) # Gain staging

        # Noise Injection
        noise = get_random_noise(data_root, sr)
        if len(noise) < max_len:
            noise = np.tile(noise, int(np.ceil(max_len / len(noise))))
        noise = noise[:max_len]
        
        target_snr = random.uniform(5, 20)
        rms_signal = np.sqrt(np.mean(mixed_audio**2) + 1e-6)
        rms_noise = np.sqrt(np.mean(noise**2) + 1e-6)
        mixed_audio += noise * (rms_signal / rms_noise) * (10 ** (-target_snr / 20))

        # Final Padding/Cropping
        if len(mixed_audio) < target_samples:
            mixed_audio = np.pad(mixed_audio, (0, target_samples - len(mixed_audio)))
        else:
            mixed_audio = mixed_audio[:target_samples]

        # Mel Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=mixed_audio, sr=sr, n_mels=128, n_fft=1024, hop_length=512)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        tensor = torch.tensor(mel_spec_db, dtype=torch.float32).unsqueeze(0)
        
        # Save Tensor
        prefix = "train" if "train" in sample_id else "val"
        save_dir = Path(config["save_dir"]) / prefix / genre
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save(tensor, save_dir / f"{sample_id}.pt")
        return True

    except Exception as e:
        return False

def generate_data():
    print(f"--- GENERATING SYNTHETIC MASHUPS ---")
    tasks = []
    buffer = 50 
    for genre in GENRES:
        for i in range(CONFIG["train_samples"] // CONFIG["num_classes"] + buffer):
            tasks.append((f"train_{genre}_{i}", genre, CONFIG))
        for i in range(CONFIG["val_samples"] // CONFIG["num_classes"] + buffer):
            tasks.append((f"val_{genre}_{i}", genre, CONFIG))

    success = 0
    with concurrent.futures.ProcessPoolExecutor(max_workers=CONFIG['num_workers']) as executor:
        futures = [executor.submit(process_single_mashup, *task) for task in tasks]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks)):
            if future.result(): success += 1
    print(f"Generated {success} tensors.")

In [8]:
# ==========================================
# 2. DATASET & DYNAMIC SPECAUGMENT
# ==========================================

class MashupDataset(Dataset):
    def __init__(self, data_dir, is_train=True):
        self.files = glob.glob(f"{data_dir}/*/*.pt")
        self.is_train = is_train
        
        # Dynamic GPU Augmentation
        self.time_masking = T.TimeMasking(time_mask_param=80)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=20)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        genre_str = Path(file_path).parent.name
        label = GENRE_TO_IDX[genre_str]
        
        mel_spec = torch.load(file_path)
        
        # Fix temporal dimension consistency
        if mel_spec.shape[-1] < TARGET_TIME_STEPS:
            pad_val = TARGET_TIME_STEPS - mel_spec.shape[-1]
            mel_spec = F.pad(mel_spec, (0, pad_val))
        else:
            mel_spec = mel_spec[:, :, :TARGET_TIME_STEPS]

        if self.is_train:
            mel_spec = self.time_masking(mel_spec)
            mel_spec = self.freq_masking(mel_spec)

        return mel_spec, torch.tensor(label, dtype=torch.long)

In [9]:
# ==========================================
# 3. DEEP CNN MODEL & FOCAL LOSS
# ==========================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()

class DeepGenreCNN(pl.LightningModule):
    def __init__(self, num_classes=10, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        
        # Deep VGG-style blocks
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.3),
            
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(4, 4), nn.Dropout2d(0.4),
            
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Dropout2d(0.5)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

        self.criterion = FocalLoss(gamma=2.0)
        self.train_f1 = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.val_f1 = MulticlassF1Score(num_classes=num_classes, average='macro')

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.train_f1(preds, y)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_macro_f1', self.train_f1, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        self.val_f1(preds, y)
        self.log('val_loss', self.criterion(logits, y), prog_bar=True)
        self.log('val_macro_f1', self.val_f1, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [10]:
# ==========================================
# 4. INFERENCE ENGINE
# ==========================================

def run_inference(model, config):
    print("--- RUNNING INFERENCE ---")
    model.eval()
    model.cuda()
    
    test_csv_path = f"{config['data_root']}/test.csv"
    test_df = pd.read_csv(test_csv_path)
    predictions = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        file_id = row['id']
        # Assuming test audio files are named by their ID in the mashups folder
        audio_path = f"{config['data_root']}/mashups/{file_id}.wav" 
        
        try:
            y, _ = librosa.load(audio_path, sr=config["sample_rate"])
            
            # Pad/Truncate to 30s
            target_samples = config["duration"] * config["sample_rate"]
            if len(y) < target_samples: y = np.pad(y, (0, target_samples - len(y)))
            else: y = y[:target_samples]
            
            mel_spec = librosa.feature.melspectrogram(y=y, sr=config["sample_rate"], n_mels=128, n_fft=1024, hop_length=512)
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
            tensor = torch.tensor(mel_spec_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).cuda()
            
            if tensor.shape[-1] < TARGET_TIME_STEPS:
                tensor = F.pad(tensor, (0, TARGET_TIME_STEPS - tensor.shape[-1]))
            else:
                tensor = tensor[:, :, :, :TARGET_TIME_STEPS]

            with torch.no_grad():
                logits = model(tensor)
                pred_idx = torch.argmax(logits, dim=1).item()
                predictions.append(IDX_TO_GENRE[pred_idx])
                
        except Exception as e:
            print(f"Failed on {file_id}: {e}")
            predictions.append("pop") # Fallback to prevent submission errors

    test_df['genre'] = predictions
    test_df[['id', 'genre']].to_csv('submission.csv', index=False)
    print("Saved predictions to submission.csv")

In [11]:
# ==========================================
# 5. EXECUTION PIPELINE
# ==========================================

if __name__ == "__main__":
    if RUN_GENERATION:
        generate_data()

    if RUN_TRAINING:
        print("--- STARTING TRAINING ---")
        train_ds = MashupDataset(f"{CONFIG['save_dir']}/train", is_train=True)
        val_ds = MashupDataset(f"{CONFIG['save_dir']}/val", is_train=False)
        
        train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=4)
        val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=4)
        
        model = DeepGenreCNN(num_classes=CONFIG["num_classes"], lr=CONFIG["lr"])
        
        checkpoint_cb = ModelCheckpoint(
            dirpath="checkpoints", filename="best-model", monitor="val_macro_f1", mode="max", save_top_k=1
        )
        early_stop_cb = EarlyStopping(monitor="val_macro_f1", patience=CONFIG["patience"], mode="max")
        
        wandb_logger = WandbLogger(project=CONFIG["project_name"], name=CONFIG["experiment_name"])
        
        trainer = pl.Trainer(
            max_epochs=CONFIG["num_epochs"],
            accelerator="gpu",
            logger=wandb_logger,
            callbacks=[checkpoint_cb, early_stop_cb],
            accumulate_grad_batches=CONFIG["accumulate_grad"],
            precision="16-mixed",
            deterministic=True
        )
        
        trainer.fit(model, train_loader, val_loader)
        
    if RUN_INFERENCE:
        # Load the best model from this run, or initialize a new one if skipping training
        checkpoint_path = "checkpoints/best-model.ckpt"
        if os.path.exists(checkpoint_path):
            print(f"Loading best checkpoint: {checkpoint_path}")
            model = DeepGenreCNN.load_from_checkpoint(checkpoint_path)
        else:
            print("No checkpoint found! Initializing untrained model for inference (Not Recommended).")
            model = DeepGenreCNN(num_classes=CONFIG["num_classes"])
            
        run_inference(model, CONFIG)

--- GENERATING SYNTHETIC MASHUPS ---


 68%|██████▊   | 3390/5000 [34:55<16:34,  1.62it/s]  


KeyboardInterrupt: 